# Credit Card Fraud Detection

In this notebook, I'll perform exploratory data analysis (EDA) and train machine learning models to detect fraudulent credit card transactions.

The dataset has:
- **V1 to V28** – PCA-transformed features (anonymized for privacy)
- **Time** – seconds elapsed since first transaction
- **Amount** – transaction amount
- **Class** – 0 = Normal, 1 = Fraud


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, f1_score, accuracy_score)
from xgboost import XGBClassifier
import joblib

plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries imported successfully!')

## 2. Load the Dataset

In [ ]:
df = pd.read_csv('../data/creditcard.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
print('Dataset Info')
print('-' * 40)
print(df.dtypes)
print('\nMissing Values:')
print(df.isnull().sum().sum(), 'total missing values')

In [ ]:
df.describe()

## 3. Exploratory Data Analysis (EDA)

### 3.1 Class Distribution (Fraud vs Normal)

In [ ]:
class_counts = df['Class'].value_counts()
print('Normal transactions:', class_counts[0])
print('Fraud  transactions:', class_counts[1])
print(f'Fraud percentage   : {class_counts[1]/len(df)*100:.2f}%')

plt.figure(figsize=(6, 4))
sns.countplot(x='Class', data=df, palette=['steelblue', 'crimson'])
plt.title('Class Distribution (0 = Normal, 1 = Fraud)')
plt.xlabel('Class')
plt.ylabel('Count')
plt.xticks([0, 1], ['Normal', 'Fraud'])
plt.tight_layout()
plt.show()

The dataset is highly imbalanced — fraud cases make up only ~2% of all transactions. This is typical in real fraud datasets.

### 3.2 Transaction Amount Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df[df['Class'] == 0]['Amount'].hist(bins=50, ax=axes[0], color='steelblue', alpha=0.7)
axes[0].set_title('Amount Distribution – Normal Transactions')
axes[0].set_xlabel('Amount')
axes[0].set_ylabel('Frequency')

df[df['Class'] == 1]['Amount'].hist(bins=50, ax=axes[1], color='crimson', alpha=0.7)
axes[1].set_title('Amount Distribution – Fraud Transactions')
axes[1].set_xlabel('Amount')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

print('Average amount – Normal:', df[df['Class']==0]['Amount'].mean().round(2))
print('Average amount – Fraud :', df[df['Class']==1]['Amount'].mean().round(2))

### 3.3 Correlation Heatmap

In [ ]:
# Correlation with the target variable
corr = df.corr()['Class'].drop('Class').sort_values()

plt.figure(figsize=(8, 6))
corr.plot(kind='barh', color=['crimson' if x > 0 else 'steelblue' for x in corr])
plt.title('Feature Correlation with Class (Fraud)')
plt.xlabel('Correlation')
plt.tight_layout()
plt.show()

### 3.4 Boxplot of Key Features

In [ ]:
# Show first 5 V features across fraud vs normal
fig, axes = plt.subplots(1, 5, figsize=(16, 4))
for i, ax in enumerate(axes):
    feature = f'V{i+1}'
    df.boxplot(column=feature, by='Class', ax=ax)
    ax.set_title(feature)
    ax.set_xlabel('Class (0=Normal, 1=Fraud)')

plt.suptitle('Distribution of V1–V5 by Class')
plt.tight_layout()
plt.show()

## 4. Data Preprocessing

In [ ]:
# Normalize Time and Amount (V1-V28 are already scaled since they are PCA features)
scaler = StandardScaler()
df['Amount'] = scaler.fit_transform(df[['Amount']])
df['Time']   = scaler.fit_transform(df[['Time']])

X = df.drop(columns=['Class'])
y = df['Class']

print('Features shape:', X.shape)
print('Target shape  :', y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set  : {len(X_train)} samples')
print(f'Test set      : {len(X_test)} samples')
print(f'Fraud in train: {y_train.sum()}')
print(f'Fraud in test : {y_test.sum()}')

## 5. Model Training

### 5.1 Logistic Regression (Baseline)

In [ ]:
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)

print('Logistic Regression Results:')
print(classification_report(y_test, lr_pred, target_names=['Normal', 'Fraud']))
print('ROC-AUC:', roc_auc_score(y_test, lr_model.predict_proba(X_test)[:, 1]).round(4))

### 5.2 Random Forest

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

print('Random Forest Results:')
print(classification_report(y_test, rf_pred, target_names=['Normal', 'Fraud']))
print('ROC-AUC:', roc_auc_score(y_test, rf_model.predict_proba(X_test)[:, 1]).round(4))

### 5.3 XGBoost

In [ ]:
xgb_model = XGBClassifier(n_estimators=100, random_state=42,
                           use_label_encoder=False, eval_metric='logloss', verbosity=0)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)

print('XGBoost Results:')
print(classification_report(y_test, xgb_pred, target_names=['Normal', 'Fraud']))
print('ROC-AUC:', roc_auc_score(y_test, xgb_model.predict_proba(X_test)[:, 1]).round(4))

## 6. Model Comparison

In [ ]:
results = {
    'Logistic Regression': {
        'model': lr_model, 'pred': lr_pred,
        'prob' : lr_model.predict_proba(X_test)[:, 1]
    },
    'Random Forest': {
        'model': rf_model, 'pred': rf_pred,
        'prob' : rf_model.predict_proba(X_test)[:, 1]
    },
    'XGBoost': {
        'model': xgb_model, 'pred': xgb_pred,
        'prob' : xgb_model.predict_proba(X_test)[:, 1]
    },
}

summary = []
for name, r in results.items():
    summary.append({
        'Model'   : name,
        'Accuracy': round(accuracy_score(y_test, r['pred']), 4),
        'F1 Score': round(f1_score(y_test, r['pred']), 4),
        'ROC-AUC' : round(roc_auc_score(y_test, r['prob']), 4),
    })

summary_df = pd.DataFrame(summary).set_index('Model')
print(summary_df)


### ROC Curve Comparison

In [ ]:
plt.figure(figsize=(8, 6))

colors = ['royalblue', 'green', 'crimson']
for (name, r), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, r['prob'])
    auc = roc_auc_score(y_test, r['prob'])
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})', color=color)

plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

### Confusion Matrix (Best Model – XGBoost)

In [ ]:
cm = confusion_matrix(y_test, xgb_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Fraud'],
            yticklabels=['Normal', 'Fraud'])
plt.title('Confusion Matrix – XGBoost')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'True Negatives  (Correctly identified as Normal): {tn}')
print(f'False Positives (Normal flagged as Fraud)       : {fp}')
print(f'False Negatives (Fraud missed)                  : {fn}')
print(f'True Positives  (Correctly identified as Fraud) : {tp}')

## 7. Feature Importance (XGBoost)

In [ ]:
importance = pd.Series(xgb_model.feature_importances_, index=X.columns)
top_features = importance.sort_values(ascending=False).head(15)

plt.figure(figsize=(8, 5))
top_features.plot(kind='bar', color='steelblue')
plt.title('Top 15 Feature Importances – XGBoost')
plt.xlabel('Feature')
plt.ylabel('Importance Score')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 8. Save Best Model

In [ ]:
import os
os.makedirs('../model', exist_ok=True)

joblib.dump(xgb_model, '../model/model.pkl')
print('Model saved to ../model/model.pkl')

## Conclusion

In this project, I trained three models to detect credit card fraud:

| Model               | F1 Score | ROC-AUC |
|---------------------|----------|---------|
| Logistic Regression | Low      | ~0.95   |
| Random Forest       | Medium   | ~0.98   |
| **XGBoost**         | **Best** | **~0.99** |

XGBoost performed the best overall. The main challenge is the class imbalance — fraud cases are very rare, so the model needs to be good at catching them without too many false alarms.

For further improvement, we could try SMOTE (oversampling), cost-sensitive learning, or ensemble stacking.
